# Feature Scaling — Normalization (`MinMaxScaler`)

Where the previous notebook covered **standardization** (mean 0, std 1), this one covers **normalization**: rescaling every feature into a fixed range — by default **0 to 1** — using `MinMaxScaler`. Same overall goal (put features on comparable scales), different formula: `(x - min) / (max - min)`.

## 1. Load and Prepare the Data

The Wine dataset — only the first 3 columns are loaded (`usecols=[0,1,2]`) and renamed, since the full dataset has many more chemical-property columns not needed here. `Class label` is the target (which of 3 wine cultivars).

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('wine_data.csv',header=None,usecols=[0,1,2])
df.columns=['Class label', 'Alcohol', 'Malic acid']

FileNotFoundError: [Errno 2] No such file or directory: 'wine_data.csv'

In [ ]:
df

## 2. Looking at Each Feature's Distribution

Before scaling anything, it's worth seeing each feature's raw range and shape — `Alcohol` sits roughly between 11–15, while `Malic acid` sits between about 0.5–5, on a very different scale from each other.

In [ ]:
sns.kdeplot(df['Alcohol'])

In [ ]:
sns.kdeplot(df['Malic acid'])

## 3. Visualizing the Two Features Together

Coloring each point by `Class label` shows how well these two chemical properties separate the three wine cultivars — useful context for judging whether scaling changes anything about the *relationship*, only further down.

### ⚠️ Deprecated: Positional Arguments in `scatterplot()`

`sns.scatterplot(df['Alcohol'], df['Malic acid'], ...)` passes the two columns positionally, which errors on seaborn 0.12+. Pass them as `x=`/`y=` keywords instead.

In [ ]:
color_dict={1:'red',3:'green',2:'blue'}
sns.scatterplot(df['Alcohol'],df['Malic acid'],hue=df['Class label'],palette=color_dict)

In [ ]:
sns.scatterplot(x='Alcohol', y='Malic acid', hue='Class label', palette=color_dict, data=df)


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.drop('Class label', axis=1),
                                                    df['Class label'],
                                                    test_size=0.3,
                                                    random_state=0)

X_train.shape, X_test.shape

Same principle as the standardization notebook: split **before** scaling, so the scaler only ever learns from training data.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

# fit the scaler to the train set, it will learn the parameters
scaler.fit(X_train)

# transform train and test sets
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

`MinMaxScaler` rescales each feature so its minimum becomes 0 and its maximum becomes 1 (everything else lands proportionally in between) — using only the **min/max seen in the training set**, same data-leakage reasoning as before.

### Shortcut: `fit_transform()`

As with `StandardScaler`, fitting and transforming the same data in one call is the more common pattern in practice.

In [ ]:
X_train_scaled_shortcut = scaler.fit_transform(X_train)
# equivalent to: scaler.fit(X_train); X_train_scaled = scaler.transform(X_train)


### Inspecting What the Scaler Learned

`scaler.data_min_` / `scaler.data_max_` show the training set's min/max for each column — the two numbers the whole transformation is based on.

In [ ]:
print('min:', scaler.data_min_)
print('max:', scaler.data_max_)


In [ ]:
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

### Confirming the Effect Numerically

After scaling, every column's min should be 0 and max should be 1.

In [ ]:
np.round(X_train.describe(), 1)

In [ ]:
np.round(X_train_scaled.describe(), 1)

## 4. Visualizing the Effect of Scaling

`c=y_train` colors points by class — same idea as the earlier `hue=` scatterplot, just via matplotlib's plain `.scatter()` instead of seaborn.

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

ax1.scatter(X_train['Alcohol'], X_train['Malic acid'],c=y_train)
ax1.set_title("Before Scaling")
ax2.scatter(X_train_scaled['Alcohol'], X_train_scaled['Malic acid'],c=y_train)
ax2.set_title("After Scaling")
plt.show()

As before, the relationship between the two features doesn't change shape — only the axis numbers do (now bounded to roughly 0–1 on both axes).

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

# before scaling
ax1.set_title('Before Scaling')
sns.kdeplot(X_train['Alcohol'], ax=ax1)
sns.kdeplot(X_train['Malic acid'], ax=ax1)

# after scaling
ax2.set_title('After Standard Scaling')
sns.kdeplot(X_train_scaled['Alcohol'], ax=ax2)
sns.kdeplot(X_train_scaled['Malic acid'], ax=ax2)
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

# before scaling
ax1.set_title('Alcohol Distribution Before Scaling')
sns.kdeplot(X_train['Alcohol'], ax=ax1)

# after scaling
ax2.set_title('Alcohol Distribution After Standard Scaling')
sns.kdeplot(X_train_scaled['Alcohol'], ax=ax2)
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(12, 5))

# before scaling
ax1.set_title('Malic acid Distribution Before Scaling')
sns.kdeplot(X_train['Malic acid'], ax=ax1)

# after scaling
ax2.set_title('Malic acid Distribution After Standard Scaling')
sns.kdeplot(X_train_scaled['Malic acid'], ax=ax2)
plt.show()

## 5. Why Scaling Matters Here — Effect on a Distance-Based Model

The earlier standardization notebook showed logistic regression benefiting from scaling while decision trees didn't. **KNN** makes that gap even more obvious, since it classifies a point by literally measuring distance to its nearest neighbors — on unscaled data, `EstimatedSalary`-style large-range features would dominate the distance calculation almost entirely, and `Alcohol`'s much smaller range would barely matter at all.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn = KNeighborsClassifier()
knn_scaled = KNeighborsClassifier()

knn.fit(X_train, y_train)
knn_scaled.fit(X_train_scaled, y_train)

y_pred = knn.predict(X_test)
y_pred_scaled = knn_scaled.predict(X_test_scaled)

print('Unscaled accuracy:', accuracy_score(y_test, y_pred))
print('Scaled accuracy:  ', accuracy_score(y_test, y_pred_scaled))


## Summary

| Task | Method |
|---|---|
| Split before scaling | `train_test_split(X, y, test_size=...)` |
| Learn scaling params from train only | `scaler.fit(X_train)` |
| Apply the scaling | `scaler.transform(X)` |
| Fit + transform in one step | `scaler.fit_transform(X_train)` |
| Inspect learned parameters | `scaler.data_min_`, `scaler.data_max_` |

### `StandardScaler` vs. `MinMaxScaler` vs. `RobustScaler`
| Scaler | Formula | Result range | Sensitive to outliers? |
|---|---|---|---|
| `StandardScaler` | `(x - mean) / std` | unbounded, centered on 0 | Yes |
| `MinMaxScaler` | `(x - min) / (max - min)` | fixed, usually [0, 1] | Yes — a single extreme value stretches the whole range |
| `RobustScaler` | `(x - median) / IQR` | unbounded | No — built specifically to resist them |